# dlt-ibapi Quickstart Guide

This notebook demonstrates the basic usage of `dlt-ibapi` for fetching market data from Interactive Brokers.

## Prerequisites

1. IB Gateway or TWS running locally (default: localhost:7497)
2. API connections enabled in IB Gateway/TWS settings
3. `dlt-ibapi` installed: `uv add dlt-ibapi`

## Table of Contents

1. [Setup and Configuration](#1-setup-and-configuration)
2. [Fetching Historical Equity Bars](#2-fetching-historical-equity-bars)
3. [Option Chain Snapshots](#3-option-chain-snapshots)
4. [Historical Data Backfilling](#4-historical-data-backfilling)
5. [Querying Saved Data](#5-querying-saved-data)

## 1. Setup and Configuration

First, let's import the necessary libraries and configure the connection to IB Gateway.

In [ ]:
import dlt
import duckdb
from datetime import date, timedelta
from pathlib import Path

# Import dlt-ibapi resources
from dlt_ibapi import ib_historical_bars, ib_source
from dlt_ibapi.config import IBConnectionConfig

# Configure IB connection (adjust if needed)
ib_config = IBConnectionConfig(
    host="127.0.0.1",
    port=7497,  # IB Gateway paper trading port
    client_id=1,
    timeout=60,
    readonly=False,
)

print("✓ Configuration loaded")
print(f"  IB Gateway: {ib_config.host}:{ib_config.port}")

## 2. Fetching Historical Equity Bars

Let's fetch some historical bar data for a single stock (AAPL) and save it to Parquet files.

In [ ]:
# Create a DLT pipeline with Parquet storage
pipeline = dlt.pipeline(
    pipeline_name="ib_quickstart",
    destination=dlt.destinations.filesystem(bucket_url="../data"),
    dataset_name="stocks",
)

# Fetch AAPL historical bars (last 30 days, daily bars)
data = ib_historical_bars(
    symbol="AAPL",
    exchange="SMART",
    currency="USD",
    bar_size="1 day",
    duration="30 D",
    connection_config=ib_config,
)

# Run the pipeline - data saved as Parquet with Hive partitioning
info = pipeline.run(data, loader_file_format="parquet")

print("\n✓ Pipeline completed successfully!")
print(f"  Rows loaded: {info.load_packages[0].jobs[0].metrics.rows if info.load_packages else 'N/A'}")
print(f"  Data saved to: ../data/stocks/")

### Query the data immediately with DuckDB

In [ ]:
# Query the Parquet files using DuckDB
conn = duckdb.connect(":memory:")

df = conn.execute("""
    SELECT 
        symbol,
        time::DATE as date,
        open,
        high,
        low,
        close,
        volume
    FROM parquet_scan('../data/stocks/**/*.parquet', hive_partitioning=true)
    WHERE symbol = 'AAPL'
    ORDER BY time DESC
    LIMIT 10
""").df()

print("\nLatest 10 bars for AAPL:")
df

## 3. Option Chain Snapshots

Capture a snapshot of available option contracts for a symbol.

In [ ]:
from dlt_ibapi.backfill.resources import snapshot_option_chain

# Create pipeline for options data
options_pipeline = dlt.pipeline(
    pipeline_name="ib_options",
    destination=dlt.destinations.filesystem(bucket_url="../data"),
    dataset_name="options",
)

# Capture option chain snapshot for SPY
snapshot = snapshot_option_chain(
    symbol="SPY",
    database_path="../data",
    dataset_name="options",
    connection_config=ib_config,
)

info = options_pipeline.run(snapshot, loader_file_format="parquet")

print("\n✓ Option chain snapshot saved!")
print(f"  Data saved to: ../data/options/option_chain_snapshot/")

### View available option expirations

In [ ]:
# Query option chain snapshot
snapshot_df = conn.execute("""
    SELECT 
        underlying,
        as_of,
        exchange,
        trading_class,
        expiration_count,
        strike_count
    FROM parquet_scan('../data/options/option_chain_snapshot/**/*.parquet', hive_partitioning=true)
    WHERE underlying = 'SPY'
    ORDER BY as_of DESC
    LIMIT 5
""").df()

print("\nOption chain snapshots for SPY:")
snapshot_df

## 4. Historical Data Backfilling

Backfill historical data with gap detection (only fetches missing dates).

In [ ]:
from dlt_ibapi.backfill.resources import backfill_equity_bars

# Backfill MSFT daily bars for the last 30 days
backfill_data = backfill_equity_bars(
    symbol="MSFT",
    database_path="../data",
    dataset_name="stocks",
    connection_config=ib_config,
    start_date=date.today() - timedelta(days=30),
    end_date=date.today(),
    bar_size="1 day",
)

info = pipeline.run(backfill_data, loader_file_format="parquet")

print("\n✓ Backfill completed!")
print("  Note: Gap detection ensures only missing dates are fetched")

### Re-run backfill (demonstrates idempotency)

In [ ]:
# Run the same backfill again - it should skip all dates (gap detection)
backfill_data_2 = backfill_equity_bars(
    symbol="MSFT",
    database_path="../data",
    dataset_name="stocks",
    connection_config=ib_config,
    start_date=date.today() - timedelta(days=30),
    end_date=date.today(),
    bar_size="1 day",
)

info = pipeline.run(backfill_data_2, loader_file_format="parquet")

print("\n✓ Second backfill completed!")
print("  Should have skipped most/all dates (idempotent)")

## 5. Querying Saved Data

Query all the data we've collected using DuckDB.

In [ ]:
# Get summary of all symbols
summary_df = conn.execute("""
    SELECT 
        symbol,
        bar_size,
        COUNT(*) as bar_count,
        MIN(time::DATE) as first_date,
        MAX(time::DATE) as last_date,
        ROUND(AVG(volume), 0) as avg_volume
    FROM parquet_scan('../data/stocks/**/*.parquet', hive_partitioning=true)
    GROUP BY symbol, bar_size
    ORDER BY symbol
""").df()

print("\nSummary of collected data:")
summary_df

### Visualize price data

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Get AAPL data for plotting
aapl_df = conn.execute("""
    SELECT 
        time::DATE as date,
        close
    FROM parquet_scan('../data/stocks/**/*.parquet', hive_partitioning=true)
    WHERE symbol = 'AAPL'
    ORDER BY time
""").df()

# Plot closing prices
plt.figure(figsize=(12, 6))
plt.plot(aapl_df['date'], aapl_df['close'], marker='o', linewidth=2)
plt.title('AAPL Closing Prices (Last 30 Days)', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Close Price ($)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"\n✓ Plotted {len(aapl_df)} data points for AAPL")

## Summary

In this quickstart notebook, you learned how to:

1. ✓ Configure connection to IB Gateway/TWS
2. ✓ Fetch historical equity bars using `ib_historical_bars`
3. ✓ Capture option chain snapshots
4. ✓ Backfill historical data with gap detection
5. ✓ Query saved Parquet data with DuckDB
6. ✓ Visualize market data

### Next Steps

- Check out `02_reading_data.ipynb` to learn more about querying Parquet data
- Explore the `examples/` directory for more advanced usage
- Read the documentation at `docs/BACKFILL_GUIDE.md`

### Data Storage

All data is stored as Parquet files with Hive-style partitioning:

```
data/
├── stocks/
│   └── date=2025-10-20/
│       ├── symbol=AAPL/*.parquet
│       └── symbol=MSFT/*.parquet
└── options/
    └── option_chain_snapshot/
        └── date=2025-10-20/
            └── underlying=SPY/*.parquet
```

Benefits:
- 10x better compression vs. raw databases
- Predicate pushdown for fast filtering
- Cloud storage ready (S3, GCS, Azure)